In [0]:
from pyspark.sql.functions import col, trim, current_timestamp

In [0]:
%sql
CREATE TABLE IF NOT EXISTS bronze.b3_cotahist (
    origin_file_name          STRING,
    origin_code               STRING,
    origin_file_creation_date STRING,
    data_loaded_timestamp     TIMESTAMP,
    trading_session_date      STRING,
    bdi_code                  STRING,
    ticker_trading_code       STRING,
    market_code               STRING,
    business_name             STRING,
    ticker_specification      STRING,
    fixed_term_days           STRING,
    trading_currency          STRING,
    opening_price             STRING,
    closing_price             STRING,
    max_price                 STRING,
    min_price                 STRING,
    average_price             STRING,
    best_buy_offer            STRING,
    best_sell_offer           STRING,
    total_number_of_deals     STRING,
    total_quantity_of_tickers STRING,
    total_volume_of_tickers   STRING,
    exercise_price            STRING,
    price_correction_idicator STRING,
    expiration_date           STRING,
    pricing_factor            STRING,
    exercise_price_points     STRING,
    internal_isin_code        STRING,
    distribution_number       STRING
)
USING DELTA
CLUSTER BY (origin_file_name, trading_session_date);

In [0]:
source_path = '/Volumes/test_playground_workspace/bronze/investing_lake/b3_stock_history/'

df_raw = (
    spark.read
        .text(source_path)
        .select(
            col("value"),
            col("_metadata.file_path").alias("origin_file_name")
        )
)

df_header = df_raw.filter(col('value').substr(1,2) == '00')
df_body = df_raw.filter(col('value').substr(1,2) == '01')

df_header = df_header.withColumns(
        {
            'origin_code': trim(col('value').substr(16, 8)),
            'origin_file_creation_date': trim(col('value').substr(24, 8)),
            'data_loaded_timestamp': current_timestamp()
        }
    ).drop('value')

df_body = (
    df_body
        .withColumns(
            {
                'trading_session_date': trim(col('value').substr(3,8)),
                'bdi_code': trim(col('value').substr(11, 2)),
                'ticker_trading_code': trim(col('value').substr(13, 12)),
                'market_code': trim(col('value').substr(25, 3)),
                'business_name': trim(col('value').substr(28, 12)),
                'ticker_specification': trim(col('value').substr(40, 10)),
                'fixed_term_days': trim(col('value').substr(50, 3)),
                'trading_currency': trim(col('value').substr(53, 4)),
                'opening_price': trim(col('value').substr(57, 13)),
                'closing_price': trim(col('value').substr(109, 13)),
                'max_price': trim(col('value').substr(70, 13)),
                'min_price': trim(col('value').substr(83, 13)),
                'average_price': trim(col('value').substr(96, 13)),
                'best_buy_offer': trim(col('value').substr(122, 13)),
                'best_sell_offer': trim(col('value').substr(135, 13)),
                'total_number_of_deals': trim(col('value').substr(148, 5)),
                'total_quantity_of_tickers': trim(col('value').substr(153, 18)),
                'total_volume_of_tickers': trim(col('value').substr(171, 18)),
                'exercise_price': trim(col('value').substr(189, 13)),
                'price_correction_idicator': trim(col('value').substr(202, 1)),
                'expiration_date': trim(col('value').substr(203, 8)),
                'pricing_factor': trim(col('value').substr(211, 7)),
                'exercise_price_points': trim(col('value').substr(218, 13)),
                'internal_isin_code': trim(col('value').substr(231, 12)),
                'distribution_number': trim(col('value').substr(243, 3))
            }
        )
        .drop('value')
    )

df_final = df_header.join(df_body, on='origin_file_name', how='inner')

In [0]:
# O número de registros de cada arquivo no df_final deve ser igual ao número de linhas no df_body para cada arquio de origem

final_row_count = (
    df_final
        .select('origin_file_name')
        .groupBy('origin_file_name')
        .count()
        .withColumnRenamed('count', 'final_row_count')
)
body_row_count = (
    df_body
        .select('origin_file_name')
        .groupBy('origin_file_name')
        .count()
        .withColumnRenamed('count', 'body_row_count')
)

display(final_row_count.join(body_row_count, on='origin_file_name', how='inner'))


In [0]:
(
    df_final.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("bronze.b3_cotahist")
)